# 3b. SBI Model Comparison: BE vs SC

| Part | Data | Fitting target | CV protocol | Cost |
|------|------|---------------|-------------|------|
| **1** | Expert only | Broad stats (SNPE) → predict UM | 2-fold × 64, ANOVA | ~1h total |
| **2** | Expert only | UM directly (per-animal SNPE) | 2-fold × 64, ANOVA | ~20 min/animal |
| **3** | All sessions | Broad stats (SNPE) → predict UM | 2-fold × 64, ANOVA | ~1h total |

In [ ]:
%matplotlib inline
import os, sys
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from behav_utils.data.structures import AnimalData, FittingData
from behav_utils.data.loading import load_experiment

from inference.comparison import (
	select_expert_fitting_data, select_all_fitting_data,
	estimate_timing, print_timing_report,
	train_amortised_snpe, train_per_animal_snpe,
	condition_on_animal, cv_um_comparison, compare_models,
	run_animal_pipeline, run_animal_pipeline_part2,
	simulate_example_session,
)
from plotting.comparison import (
	plot_sbi_cv_comparison as plot_cv_comparison,
	plot_example_session
)

try:
	import torch; SBI_OK = True
	print(f"torch={torch.__version__}, {'cuda' if torch.cuda.is_available() else 'cpu'}")
except ImportError:
	SBI_OK = False; print("SBI not available")

## 0. Configuration

In [ ]:
CONFIG_PATH = '/Users/Serkan/Desktop/pro/PhD/main/repos/sound_categorisation/config.yaml'

# ── Session selection ─────────────────────────────────────────────────
STAGE = 'Full_Task_Cont'
DISTRIBUTION = 'Uniform'
MIN_VALID_TRIALS = 30
TARGET_ANIMAL = None       # e.g. 'SS05' or None = all

# ── Expert definition ─────────────────────────────────────────────────
EXPERT_MIN_ACCURACY = 0.70
EXPERT_LAST_FRACTION = 0.50

# ── Stats for Parts 1 & 3 (NO update_matrix — sequence-independent) ──
SBI_STATS = [
	'accuracy', 'psychometric', 'psychometric_gof',
	'recency', 'stimulus_recency', 'win_stay', 'lose_shift',
	'stimulus_sensitivity', 'hard_easy_ratio',
	'logistic_history',
]

# ── Stats for Part 2 (WITH update_matrix — uses real stimuli) ────────
SBI_STATS_WITH_UM = ['update_matrix']

# ── SNPE training ─────────────────────────────────────────────────────
N_SBI_SIMS = 50_000        # Amortised (Parts 1 & 3)
N_SBI_SIMS_P2 = 10_000    # Per-animal (Part 2) — less needed
N_GENERIC_TRIALS = 2500    # Generic stimulus sequence length
EXPERT_BURN_IN = 1000
FULL_BURN_IN = 0
SEED = 42

# ── Cross-validation ──────────────────────────────────────────────────
N_CV_FOLDS = 2
N_CV_REPEATS = 64

# ── Plotting ──────────────────────────────────────────────────────────
BE_COL = 'steelblue'
SC_COL = 'darkorange'

In [ ]:
experiment = load_experiment(CONFIG_PATH)

animals = []
for a in experiment.animals:
	animal = experiment.get_animal(a)
	try:
		fd = select_expert_fitting_data(animal, STAGE, DISTRIBUTION,
									 EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION)
		n_exp = fd.n_sessions
	except ValueError:
		n_exp = 0
	n_all = len(animal.get_sessions(stage=STAGE, distribution=DISTRIBUTION))
	ok = n_exp >= 5
	print(f"  {animal.animal_id}: {n_all} total, {n_exp} expert {'  ok' if ok else '  skip'}")
	if ok and (TARGET_ANIMAL is None or animal.animal_id == TARGET_ANIMAL):
		animals.append(animal)

print(f"\nFitting: {[a.animal_id for a in animals]}")

## 0b. Timing estimation

**Run this before committing.** Shows per-simulation cost and projected
total time for each part.

In [ ]:
# Part 1 & 3: amortised stats (no UM)
t_amort = estimate_timing(
	SBI_STATS, n_trials=N_GENERIC_TRIALS,
	burn_in=EXPERT_BURN_IN, n_sbi_sims=N_SBI_SIMS,
)
print_timing_report(t_amort, N_SBI_SIMS, label='Parts 1 & 3 (amortised, no UM)')

# Part 2: per-animal with UM
t_p2 = estimate_timing(
	SBI_STATS_WITH_UM, n_trials=N_GENERIC_TRIALS,
	burn_in=EXPERT_BURN_IN, n_sbi_sims=N_SBI_SIMS_P2,
)
print_timing_report(t_p2, N_SBI_SIMS_P2, n_animals=len(animals),
					label='Part 2 (per-animal, with UM)')

# Full trajectory (burn_in=0)
t_full = estimate_timing(
	SBI_STATS, n_trials=5000,
	burn_in=FULL_BURN_IN, n_sbi_sims=N_SBI_SIMS,
)
print_timing_report(t_full, N_SBI_SIMS, label='Part 3 (full trajectory, no UM)')

# Total estimate
p1_h = t_amort['be']['total_hours'] + t_amort['sc']['total_hours']
p2_h = (t_p2['be']['total_hours'] + t_p2['sc']['total_hours']) * len(animals)
p3_h = t_full['be']['total_hours'] + t_full['sc']['total_hours']
print(f"\n{'='*60}")
print(f"  TOTAL ESTIMATED TIME")
print(f"  Part 1 (train once):       {p1_h:.1f}h")
print(f"  Part 2 ({len(animals)} animals):     {p2_h:.1f}h")
print(f"  Part 3 (train once):       {p3_h:.1f}h")
print(f"  CV loops (all parts):      ~0.5h")
print(f"  ─────────────────────────────")
print(f"  Grand total:               ~{p1_h + p2_h + p3_h + 0.5:.1f}h")
print(f"{'='*60}")
print(f"\nIf Part 2 is too slow, reduce N_SBI_SIMS_P2 or drop UM from stats.")
print(f"If everything is too slow, use SBI_STATS = ['accuracy', 'psychometric',")
print(f"    'recency', 'win_stay', 'stimulus_sensitivity'] (8 dims, ~10x faster).")

---
# Part 1: Expert — Our Approach

Amortised SNPE on broad stats → predict UM on held-out folds.

In [ ]:
# 1.1 Train amortised SNPE (once for all animals)
if SBI_OK:
	be_snpe_exp = train_amortised_snpe(
		'be', SBI_STATS, N_SBI_SIMS, N_GENERIC_TRIALS, EXPERT_BURN_IN, SEED)
	sc_snpe_exp = train_amortised_snpe(
		'sc', SBI_STATS, N_SBI_SIMS, N_GENERIC_TRIALS, EXPERT_BURN_IN, SEED+1)

In [ ]:
def plot_cv_comparison_2(
	be_cv, sc_cv, comparison,
	animal_id: str, title_suffix: str = '',
	be_colour: str = 'steelblue', sc_colour: str = 'darkorange',
):
	"""Paired violin + scatter plot of CV test errors."""
	import matplotlib.pyplot as plt

	fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
	n = min(len(be_cv['test_errors']), len(sc_cv['test_errors']))

	ax = axes[0]
	for i in range(n):
		ax.plot([0, 1], [be_cv['test_errors'][i], sc_cv['test_errors'][i]],
				'o-', color='grey', alpha=0.15, markersize=3)
	parts = ax.violinplot([be_cv['test_errors'], sc_cv['test_errors']],
						  positions=[0, 1], showmedians=True)
	for pc, col in zip(parts['bodies'], [be_colour, sc_colour]):
		pc.set_facecolor(col); pc.set_alpha(0.3)
	ax.set_xticks([0, 1]); ax.set_xticklabels(['BE', 'SC'])
	ax.set_ylabel('Test UM MSE')
	ax.set_title(f'p={comparison["p_value"]:.3g}, winner={comparison["winner"]}')

	ax = axes[1]
	ax.scatter(be_cv['test_errors'][:n], sc_cv['test_errors'][:n], alpha=0.4, s=20, c='grey')
	lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
	ax.plot([0, lim], [0, lim], 'k--', alpha=0.3)
	ax.set_xlabel('BE test error'); ax.set_ylabel('SC test error')
	ax.set_title(f'BE={comparison["be_mean"]:.5f}, SC={comparison["sc_mean"]:.5f}')
	ax.set_aspect('equal')

	fig.suptitle(f'{animal_id} — CV {title_suffix}', fontsize=13, fontweight='bold')
	plt.tight_layout()
	return fig

In [ ]:
# 1.2 Per-animal: condition + CV + example session
p1 = []
if SBI_OK:
	for animal in animals:
		expert_fd = select_expert_fitting_data(
			animal, STAGE, DISTRIBUTION, EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION)
		r = run_animal_pipeline(
			animal, expert_fd, be_snpe_exp, sc_snpe_exp,
			n_cv_repeats=N_CV_REPEATS, seed=SEED)
		p1.append(r)

		plot_cv_comparison(r['be_cv'], r['sc_cv'],
			compare_models(r['be_cv'], r['sc_cv']),
			animal.animal_id, '(Part 1: our approach)')

		ex = simulate_example_session(
			animal, -3, r['be_params'], r['sc_params'],
			STAGE, DISTRIBUTION, EXPERT_BURN_IN)
		plot_example_session(ex, animal.animal_id)

In [ ]:
df1

In [ ]:
df3

In [ ]:
df2['n_sessions'] = df1['n_sessions']
df2['n_trials'] = df1['n_trials']


In [ ]:
# 1.3 Summary
if p1:
	df1 = pd.DataFrame([{k: r[k] for k in
		['animal_id','n_sessions','n_trials','winner','p','be_mean','sc_mean']
	} for r in p1])
	print("Part 1 — Expert, Our Approach:")
	print(df1.to_string(index=False))
	print(f"Tally: {df1['winner'].value_counts().to_dict()}")

In [ ]:
# Save Part 1 results
import pickle

def _make_portable(snpe_result):
	"""Strip unpicklable objects from SNPE result."""
	return {k: v for k, v in snpe_result.items() 
			if k not in ('simulator', 'sbi_sim', 'prior')}

with open('results_part1.pkl', 'wb') as f:
	pickle.dump({
		'p1': p1,
		'be_snpe_exp': _make_portable(be_snpe_exp),
		'sc_snpe_exp': _make_portable(sc_snpe_exp),
	}, f)
print("Part 1 saved → results_part1.pkl")

---
# Part 2: Expert — Manuscript Approach

Per-animal SNPE with `update_matrix` in the observation vector.
Uses each animal's real stimulus sequence — no sequence mismatch.
Same CV protocol as the manuscript (2-fold × 64, ANOVA on UM MSE).

The difference from grid search: SNPE finds parameters more efficiently,
but the fitting target (reproduce the UM) and evaluation protocol are identical.

In [ ]:
# # 2.1 Per-animal: train SNPE with UM + CV
# p2 = []
# for animal in animals:
# 	expert_fd = select_expert_fitting_data(
# 		animal, STAGE, DISTRIBUTION, EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION)
# 	r = run_animal_pipeline_part2(
# 		animal, expert_fd, SBI_STATS_WITH_UM,
# 		n_sbi_sims=N_SBI_SIMS_P2, n_cv_repeats=N_CV_REPEATS,
# 		burn_in=EXPERT_BURN_IN, seed=SEED)
# 	p2.append(r)

# 	plot_cv_comparison(r['be_cv'], r['sc_cv'],
# 		compare_models(r['be_cv'], r['sc_cv']),
# 		animal.animal_id, '(Part 2: manuscript approach)')

In [ ]:
import pickle
with open('results_part2.pkl', 'rb') as f:
	p2 = pickle.load(f)
print(f"Loaded {len(p2)} animals")

In [ ]:
# 2.2 Summary + agreement with Part 1
if p2:
	df2 = pd.DataFrame([{k: r[k] for k in
		['animal_id','winner','p','be_mean','sc_mean']
	} for r in p2])
	print("Part 2 — Expert, Manuscript Approach:")
	print(df2.to_string(index=False))
	print(f"Tally: {df2['winner'].value_counts().to_dict()}")

	if p1:
		agree = pd.DataFrame({
			'animal': [r['animal_id'] for r in p1],
			'Part1': [r['winner'] for r in p1],
			'Part2': [r['winner'] for r in p2],
		})
		agree['match'] = agree['Part1'] == agree['Part2']
		print(f"\nAgreement: {agree['match'].mean():.0%}")
		print(agree.to_string(index=False))

In [ ]:
# # Save Part 2 results
# with open('results_part2.pkl', 'wb') as f:
#     pickle.dump({'p2': p2}, f)
# print("Part 2 saved → results_part2.pkl")

---
# Part 3: Full Trajectory — Our Approach

All Uniform sessions (including learning). Burn-in = 0.

In [ ]:
# 3.1 Train amortised SNPE (once)
if SBI_OK:
	be_snpe_full = train_amortised_snpe(
		'be', SBI_STATS, N_SBI_SIMS, 5000, FULL_BURN_IN, SEED+10)
	sc_snpe_full = train_amortised_snpe(
		'sc', SBI_STATS, N_SBI_SIMS, 5000, FULL_BURN_IN, SEED+11)

In [ ]:
# 3.2 Per-animal
p3 = []
if SBI_OK:
	for animal in animals:
		full_fd = select_all_fitting_data(animal, STAGE, DISTRIBUTION, MIN_VALID_TRIALS)
		r = run_animal_pipeline(
			animal, full_fd, be_snpe_full, sc_snpe_full,
			n_cv_repeats=N_CV_REPEATS, seed=SEED)
		p3.append(r)

		plot_cv_comparison(r['be_cv'], r['sc_cv'],
			compare_models(r['be_cv'], r['sc_cv']),
			animal.animal_id, '(Part 3: full trajectory)')

In [ ]:
if p3:
	df3 = pd.DataFrame([{k: r[k] for k in
		['animal_id','n_sessions','n_trials','winner','p','be_mean','sc_mean']
	} for r in p3])
	print("Part 3 — Full Trajectory:")
	print(df3.to_string(index=False))
	print(f"Tally: {df3['winner'].value_counts().to_dict()}")

In [ ]:
# Save Part 3 results
def _make_portable(snpe_result):
	"""Strip unpicklable objects from SNPE result."""
	return {k: v for k, v in snpe_result.items() 
			if k not in ('simulator', 'sbi_sim', 'prior')}

with open('results_part3.pkl', 'wb') as f:
	pickle.dump({
		'p3': p3,
		'be_snpe_full': _make_portable(be_snpe_full),
		'sc_snpe_full': _make_portable(sc_snpe_full),
	}, f)
print("Part 3 saved → results_part3.pkl")

---
# Grand Summary

In [ ]:
if p1 and p2 and p3:
	grand = pd.DataFrame({
		'animal': [r['animal_id'] for r in p1],
		'P1_expert_ours': [r['winner'] for r in p1],
		'P2_expert_manuscript': [r['winner'] for r in p2],
		'P3_full_ours': [r['winner'] for r in p3],
	})
	print("Model assignment per animal:\n")
	print(grand.to_string(index=False))

	print("\nTallies:")
	for col in grand.columns[1:]:
		print(f"  {col}: {grand[col].value_counts().to_dict()}")

	print(f"\nP1 vs P2 agreement: {(grand.iloc[:,1]==grand.iloc[:,2]).mean():.0%}")
	print(f"P1 vs P3 agreement: {(grand.iloc[:,1]==grand.iloc[:,3]).mean():.0%}")
	print(f"P2 vs P3 agreement: {(grand.iloc[:,2]==grand.iloc[:,3]).mean():.0%}")

	# Scatter: Part 1 vs Part 2 error ratios
	fig, ax = plt.subplots(figsize=(6, 6))
	p1_ratio = np.array([r['be_mean'] for r in p1]) / np.array([r['sc_mean'] for r in p1])
	p2_ratio = np.array([r['be_mean'] for r in p2]) / np.array([r['sc_mean'] for r in p2])
	ax.scatter(p1_ratio, p2_ratio, s=60, alpha=0.7)
	for i, r in enumerate(p1):
		ax.annotate(r['animal_id'], (p1_ratio[i], p2_ratio[i]), fontsize=8)
	ax.axhline(1, color='grey', ls='--', alpha=0.3)
	ax.axvline(1, color='grey', ls='--', alpha=0.3)
	ax.set_xlabel('Part 1: BE/SC error ratio')
	ax.set_ylabel('Part 2: BE/SC error ratio')
	ax.set_title('Method agreement (< 1 = BE wins)')
	plt.tight_layout(); plt.show()

---
# Session-by-Session Visualisation

For each animal, shows:
- **UM grid**: rows = sessions, columns = Real | BE | SC
- **Psychometric grid**: Real (black) + BE (blue) + SC (orange) overlaid
- **Pooled UM comparison**: averaged across all sessions

Uses Part 1 (our approach) params by default. Change `results_to_use`
to `p2` for manuscript-approach params.

In [ ]:
from inference.comparison import condition_on_animal, cv_um_comparison, compare_models
from plotting.comparison import plot_sbi_cv_comparison as plot_cv_comparison

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
results_to_use = p1        # Change to p2 for manuscript params
VIS_BURN_IN = EXPERT_BURN_IN
VIS_N_REPS = 20            # Stochastic realisations per model per session
VIS_MAX_SESSIONS = 20      # Cap for very long session lists

In [ ]:
# Generate session-by-session data for each animal
all_session_data = {}

for r in results_to_use:
	aid = r['animal_id']
	animal = experiment.get_animal(aid)
	print(f"\nSimulating all sessions for {aid}...")

	sdata = simulate_all_sessions(
		animal,
		be_params=r['be_params'],
		sc_params=r['sc_params'],
		stage=STAGE, distribution=DISTRIBUTION,
		min_accuracy=EXPERT_MIN_ACCURACY,
		last_fraction=EXPERT_LAST_FRACTION,
		burn_in=VIS_BURN_IN,
		n_reps=VIS_N_REPS,
		seed=SEED,
	)
	all_session_data[aid] = sdata
	print(f"  {len(sdata)} sessions simulated")

In [ ]:
# # Plot for each animal
# for aid, sdata in all_session_data.items():
#     print(f"\n{'='*60}")
#     print(f"  {aid}")
#     print(f"{'='*60}")

#     # Pooled UM comparison (most important — manuscript-style figure)
#     plot_pooled_um_comparison(sdata, aid)

#     # Psychometric curves (compact grid)
#     plot_session_by_session_psychometric(
#         sdata, aid, max_sessions=VIS_MAX_SESSIONS)

#     # # Full UM grid (large figure — scroll down)
#     # plot_session_by_session_um(
#     #     sdata, aid, max_sessions=VIS_MAX_SESSIONS)

In [ ]:
# # Per-session MSE trajectory: how does model fit evolve across sessions?
# for aid, sdata in all_session_data.items():
#     if not sdata:
#         continue

#     sess_idx = [d['session_idx'] for d in sdata]
#     be_mses = [d['be_um_mse'] for d in sdata]
#     sc_mses = [d['sc_um_mse'] for d in sdata]
#     accs = [d['accuracy'] for d in sdata]

#     fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

#     ax = axes[0]
#     ax.plot(sess_idx, be_mses, 'o-', color=BE_COL, label='BE', markersize=5)
#     ax.plot(sess_idx, sc_mses, 's-', color=SC_COL, label='SC', markersize=5)
#     ax.set_ylabel('UM MSE')
#     ax.set_title(f'{aid} — Per-session UM MSE')
#     ax.legend(fontsize=9)
#     ax.grid(True, alpha=0.15)

#     ax = axes[1]
#     ax.plot(sess_idx, accs, 'ko-', markersize=5)
#     ax.axhline(EXPERT_MIN_ACCURACY, color='red', ls='--', alpha=0.4,
#                label=f'Expert threshold ({EXPERT_MIN_ACCURACY})')
#     ax.set_ylabel('Accuracy')
#     ax.set_xlabel('Session index')
#     ax.legend(fontsize=9)
#     ax.grid(True, alpha=0.15)

#     plt.tight_layout()
#     plt.show()

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
results_to_use_2 = p2        # Change to p2 for manuscript params
VIS_BURN_IN = EXPERT_BURN_IN
VIS_N_REPS = 20            # Stochastic realisations per model per session
VIS_MAX_SESSIONS = 20      # Cap for very long session lists

# Generate session-by-session data for each animal
all_session_data_2 = {}

for r in results_to_use_2:
	aid = r['animal_id']
	animal = experiment.get_animal(aid)
	print(f"\nSimulating all sessions for {aid}...")

	sdata = simulate_all_sessions(
		animal,
		be_params=r['be_params'],
		sc_params=r['sc_params'],
		stage=STAGE, distribution=DISTRIBUTION,
		min_accuracy=EXPERT_MIN_ACCURACY,
		last_fraction=EXPERT_LAST_FRACTION,
		burn_in=VIS_BURN_IN,
		n_reps=VIS_N_REPS,
		seed=SEED,
	)
	all_session_data_2[aid] = sdata
	print(f"  {len(sdata)} sessions simulated")

In [ ]:
# Plot for each animal
for aid, sdata in all_session_data_2.items():
	print(f"\n{'='*60}")
	print(f"  {aid}")
	print(f"{'='*60}")

	# Pooled UM comparison (most important — manuscript-style figure)
	plot_pooled_um_comparison(sdata, aid)

	# Psychometric curves (compact grid)
	plot_session_by_session_psychometric(
		sdata, aid, max_sessions=VIS_MAX_SESSIONS)

	# # Full UM grid (large figure — scroll down)
	# plot_session_by_session_um(
	#     sdata, aid, max_sessions=VIS_MAX_SESSIONS)

In [ ]:
# Per-session MSE trajectory: how does model fit evolve across sessions?
for aid, sdata in all_session_data_2.items():
	if not sdata:
		continue

	sess_idx = [d['session_idx'] for d in sdata]
	be_mses = [d['be_um_mse'] for d in sdata]
	sc_mses = [d['sc_um_mse'] for d in sdata]
	accs = [d['accuracy'] for d in sdata]

	fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

	ax = axes[0]
	ax.plot(sess_idx, be_mses, 'o-', color=BE_COL, label='BE', markersize=5)
	ax.plot(sess_idx, sc_mses, 's-', color=SC_COL, label='SC', markersize=5)
	ax.set_ylabel('UM MSE')
	ax.set_title(f'{aid} — Per-session UM MSE')
	ax.legend(fontsize=9)
	ax.grid(True, alpha=0.15)

	ax = axes[1]
	ax.plot(sess_idx, accs, 'ko-', markersize=5)
	ax.axhline(EXPERT_MIN_ACCURACY, color='red', ls='--', alpha=0.4,
			   label=f'Expert threshold ({EXPERT_MIN_ACCURACY})')
	ax.set_ylabel('Accuracy')
	ax.set_xlabel('Session index')
	ax.legend(fontsize=9)
	ax.grid(True, alpha=0.15)

	plt.tight_layout()
	plt.show()

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
results_to_use_3 = p3        # Change to p2 for manuscript params
VIS_BURN_IN = EXPERT_BURN_IN
VIS_N_REPS = 20            # Stochastic realisations per model per session
VIS_MAX_SESSIONS = 20      # Cap for very long session lists

# Generate session-by-session data for each animal
all_session_data_3 = {}

for r in results_to_use_3:
	aid = r['animal_id']
	animal = experiment.get_animal(aid)
	print(f"\nSimulating all sessions for {aid}...")

	sdata = simulate_all_sessions(
		animal,
		be_params=r['be_params'],
		sc_params=r['sc_params'],
		stage=STAGE, distribution=DISTRIBUTION,
		min_accuracy=EXPERT_MIN_ACCURACY,
		last_fraction=EXPERT_LAST_FRACTION,
		burn_in=VIS_BURN_IN,
		n_reps=VIS_N_REPS,
		seed=SEED,
	)
	all_session_data_3[aid] = sdata
	print(f"  {len(sdata)} sessions simulated")

In [ ]:
# Plot for each animal
for aid, sdata in all_session_data_3.items():
	print(f"\n{'='*60}")
	print(f"  {aid}")
	print(f"{'='*60}")

	# Pooled UM comparison (most important — manuscript-style figure)
	plot_pooled_um_comparison(sdata, aid)

	# Psychometric curves (compact grid)
	plot_session_by_session_psychometric(
		sdata, aid, max_sessions=VIS_MAX_SESSIONS)

	# # Full UM grid (large figure — scroll down)
	# plot_session_by_session_um(
	#     sdata, aid, max_sessions=VIS_MAX_SESSIONS)

In [ ]:
# Per-session MSE trajectory: how does model fit evolve across sessions?
for aid, sdata in all_session_data_3.items():
	if not sdata:
		continue

	sess_idx = [d['session_idx'] for d in sdata]
	be_mses = [d['be_um_mse'] for d in sdata]
	sc_mses = [d['sc_um_mse'] for d in sdata]
	accs = [d['accuracy'] for d in sdata]

	fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

	ax = axes[0]
	ax.plot(sess_idx, be_mses, 'o-', color=BE_COL, label='BE', markersize=5)
	ax.plot(sess_idx, sc_mses, 's-', color=SC_COL, label='SC', markersize=5)
	ax.set_ylabel('UM MSE')
	ax.set_title(f'{aid} — Per-session UM MSE')
	ax.legend(fontsize=9)
	ax.grid(True, alpha=0.15)

	ax = axes[1]
	ax.plot(sess_idx, accs, 'ko-', markersize=5)
	ax.axhline(EXPERT_MIN_ACCURACY, color='red', ls='--', alpha=0.4,
			   label=f'Expert threshold ({EXPERT_MIN_ACCURACY})')
	ax.set_ylabel('Accuracy')
	ax.set_xlabel('Session index')
	ax.legend(fontsize=9)
	ax.grid(True, alpha=0.15)

	plt.tight_layout()
	plt.show()

In [ ]:
from behav_utils.analysis import fit_psychometric, compute_update_matrix, matrix_error
from behav_utils.analysis.utils import cumulative_gaussian
from behav_utils.plotting.update_matrix import plot_update_matrix as _plot_um

def collect_animal_summaries_2(
	results,
	all_session_data,
	n_bins: int = 8,
	n_bootstrap: int = 200,
	burn_in: int = 1000,
	n_reps: int = 20,
	seed: int = 42):
	"""
	For each animal, compute pooled UMs and psychometric fits
	for Real, BE, and SC by simulating with fitted params.
	"""
	from models.BE_core import BEParams, BEModel
	from models.SC_core import SCParams, SCModel

	summaries = []

	for r in results:
		aid = r['animal_id']
		sdata = all_session_data.get(aid, [])
		if not sdata:
			continue

		# Pool real data
		all_stim = np.concatenate([d['stimuli'] for d in sdata])
		all_cat = np.concatenate([d['categories'] for d in sdata])
		all_ch = np.concatenate([d['choices'] for d in sdata])

		# Real UM and psychometric
		real_um, _, _ = compute_update_matrix(all_stim, all_ch, all_cat, n_bins)
		real_psych = fit_psychometric(all_stim, all_ch,
									  n_bootstrap=n_bootstrap, seed=seed)

		# Simulate BE and SC on pooled stimuli
		be_p = BEParams(**r['be_params'])
		sc_p = SCParams(**r['sc_params'])

		be_all_choices, sc_all_choices = [], []
		be_ums, sc_ums = [], []

		for rep in range(n_reps):
			rng = np.random.default_rng(seed + rep)

			# BE
			be_state = BEModel.create_initial_state(
				params=be_p, burn_in=burn_in, seed=seed)
			be_ch, _, _, _ = BEModel.simulate_session(
				be_p, be_state, all_stim, all_cat, rng, return_history=False)
			be_all_choices.append(be_ch)
			v = ~np.isnan(be_ch)
			if v.sum() > 50:
				um, _, _ = compute_update_matrix(
					all_stim[v], be_ch[v], all_cat[v], n_bins)
				be_ums.append(um)

			# SC
			rng2 = np.random.default_rng(seed + rep + 1000)
			sc_state = SCModel.create_initial_state(
				params=sc_p, burn_in=burn_in, seed=seed)
			sc_ch, _, _, _ = SCModel.simulate_session(
				sc_p, sc_state, all_stim, all_cat, rng2, return_history=False)
			sc_all_choices.append(sc_ch)
			v = ~np.isnan(sc_ch)
			if v.sum() > 50:
				um, _, _ = compute_update_matrix(
					all_stim[v], sc_ch[v], all_cat[v], n_bins)
				sc_ums.append(um)

		# Mean UMs
		be_um = np.nanmean(be_ums, axis=0) if be_ums else np.full((n_bins, n_bins), np.nan)
		sc_um = np.nanmean(sc_ums, axis=0) if sc_ums else np.full((n_bins, n_bins), np.nan)

		# Psychometrics from averaged simulated choices
		be_mean_ch = np.nanmean(be_all_choices, axis=0)
		sc_mean_ch = np.nanmean(sc_all_choices, axis=0)
		be_psych = fit_psychometric(all_stim, be_mean_ch,
									n_bootstrap=n_bootstrap, seed=seed)
		sc_psych = fit_psychometric(all_stim, sc_mean_ch,
									n_bootstrap=n_bootstrap, seed=seed)

		summaries.append({
			'animal_id': aid,
			'real_um': real_um, 'be_um': be_um, 'sc_um': sc_um,
			'real_psych': real_psych, 'be_psych': be_psych, 'sc_psych': sc_psych,
			'be_um_mse': matrix_error(be_um, real_um),
			'sc_um_mse': matrix_error(sc_um, real_um),
		})

	return summaries

def plot_animal_grid(
	summaries,
	title: str = 'Model Comparison',
	be_colour: str = 'steelblue',
	sc_colour: str = 'darkorange',
	figscale: float = 3.0,
):
	"""
	Rows = animals, Columns = Real UM | BE UM | SC UM | Psychometric overlay.

	Psychometric column shows Real (black) + BE (blue) + SC (orange)
	with bootstrap confidence bands.
	"""
	n_animals = len(summaries)
	if n_animals == 0:
		print("No data to plot.")
		return

	# Shared UM colour scale
	all_vals = []
	for s in summaries:
		for um in [s['real_um'], s['be_um'], s['sc_um']]:
			if um is not None and not np.all(np.isnan(um)):
				all_vals.append(np.nanmax(np.abs(um)))
	vlim = max(all_vals) if all_vals else 0.3

	fig, axes = plt.subplots(
		n_animals, 4,
		figsize=(figscale * 4, figscale * n_animals),
		squeeze=False,
	)

	x_fine = np.linspace(-1.1, 1.1, 200)

	for row, s in enumerate(summaries):
		aid = s['animal_id']

		# ── Columns 0–2: Update matrices ──
		for col, (um, label, mse_key) in enumerate([
			(s['real_um'], 'Real Update Matrix', None),
			(s['be_um'], 'BE', 'be_um_mse'),
			(s['sc_um'], 'SC', 'sc_um_mse'),
		]):
			ax = axes[row, col]
			if um is not None and not np.all(np.isnan(um)):
				_plot_um(um, ax=ax, vmin=-vlim, vmax=vlim, show_colorbar=False)
			else:
				ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center')

			# Row label
			if col == 0:
				ax.set_ylabel(aid, fontsize=11, fontweight='bold')
			else:
				ax.set_ylabel('')

			# Column header
			if mse_key is None:
				ax.set_title(label, fontsize=12, fontweight='bold')
			else:
				text = f'{label} - MSE: {s[mse_key]:.4f}'
				ax.set_title(text, fontsize=12, fontweight='bold',
								color=be_colour if label == 'BE' else sc_colour)

			# # MSE annotation
			# if mse_key is not None:
			#     mse = s[mse_key]
			#     if not np.isnan(mse):
			#         ax.text(0.02, 0.98, f'MSE={mse:.4f}',
			#                 transform=ax.transAxes, fontsize=7,
			#                 va='top', ha='left',
			#                 color=be_colour if label == 'BE' else sc_colour)

			ax.tick_params(labelsize=5)
			if row < n_animals - 1:
				ax.set_xlabel('')

		# ── Column 3: Psychometric curves ──
		ax = axes[row, 3]

		for psych, colour, label, lw in [
			(s['real_psych'], 'black', 'Real', 2.5),
			(s['be_psych'], be_colour, 'BE', 1.8),
			(s['sc_psych'], sc_colour, 'SC', 1.8),
		]:
			if not psych.get('success', False):
				continue

			y = cumulative_gaussian(
				x_fine, psych['mu'], psych['sigma'],
				psych['lapse_low'], psych['lapse_high'],
			)
			ax.plot(x_fine, y, '-', color=colour, lw=lw, label=label)

			# Bootstrap CI band
			if 'y_fit_ci' in psych:
				lo, hi = psych['y_fit_ci']
				ax.fill_between(psych.get('x_fit', x_fine)[:len(lo)],
								lo, hi, color=colour, alpha=0.15)

		ax.axhline(0.5, color='grey', ls='--', alpha=0.3, lw=0.5)
		ax.axvline(0, color='grey', ls='--', alpha=0.3, lw=0.5)
		ax.set_xlim(-1.15, 1.15)
		ax.set_ylim(-0.05, 1.05)
		ax.tick_params(labelsize=7)

		if row == 0:
			ax.set_title('Psychometric', fontsize=12, fontweight='bold')
			ax.legend(fontsize=7, loc='lower right')
		if row == n_animals - 1:
			ax.set_xlabel('Stimulus', fontsize=9)

	fig.suptitle(title, fontsize=15, fontweight='bold', y=1.01)
	plt.tight_layout()
	plt.show()

	return fig

In [ ]:
"""
Summary comparison figures: boxplot-based.

- plot_param_comparison_box: psychometric params as boxplots
- compute_session_metrics: simulates choices and computes agreement/accuracy/UM MSE
- plot_animal_summary_boxplots: per-animal (dots = sessions)
- plot_group_summary_boxplots: group-level (dots = animals)

Put next to the notebook.
"""

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Optional
from scipy.stats import wilcoxon

from behav_utils.analysis.update_matrix import compute_update_matrix, matrix_error
from behav_utils.analysis.psychometry import fit_psychometric
from behav_utils.analysis.utils import cumulative_gaussian


def _significance_stars(p):
	if p >= 0.05: return 'ns'
	elif p < 0.001: return '***'
	elif p < 0.01: return '**'
	else: return '*'


def _draw_sig_bracket(ax, x1, x2, y, stars, step=0.02):
	ax.plot([x1, x1, x2, x2], [y, y + step, y + step, y], color='black', lw=0.8)
	ax.text((x1 + x2) / 2, y + step * 1.2, stars, ha='center', va='bottom', fontsize=9,
			fontweight='bold' if stars != 'ns' else 'normal',
			color='black' if stars != 'ns' else 'grey')


def _boxplot_with_dots_and_tests(ax, data_dict, colours, positions, labels,
								  pairs, ylabel='', title='', dot_size=20,
								  jitter_width=0.1, seed=42):
	rng = np.random.default_rng(seed)
	n_items = len(list(data_dict.values())[0])

	box_data = [data_dict[l][~np.isnan(data_dict[l])] for l in labels]
	pos_list = [positions[l] for l in labels]

	bp = ax.boxplot(box_data, positions=pos_list, widths=0.45, patch_artist=True,
					showfliers=False, medianprops=dict(color='black', lw=1.5))
	for patch, l in zip(bp['boxes'], labels):
		patch.set_facecolor(colours[l]); patch.set_alpha(0.3)
		patch.set_edgecolor(colours[l]); patch.set_linewidth(1.5)

	for i in range(n_items):
		vals = [data_dict[l][i] for l in labels]
		if not any(np.isnan(vals)):
			ax.plot([positions[l] for l in labels], vals, '-', color='grey', alpha=0.25, lw=0.7)

	for l in labels:
		arr = data_dict[l]; valid = ~np.isnan(arr); x = positions[l]
		jitter = rng.uniform(-jitter_width, jitter_width, valid.sum())
		ax.scatter(np.full(valid.sum(), x) + jitter, arr[valid], color=colours[l],
				   s=dot_size, alpha=0.6, edgecolors='white', linewidths=0.5, zorder=5)

	all_valid = [data_dict[l] for l in labels]
	y_max = max(np.nanmax(a) for a in all_valid if np.any(~np.isnan(a)))
	y_range = y_max - ax.get_ylim()[0] if ax.get_ylim()[0] < y_max else y_max
	y_step = max(y_range * 0.06, 0.01)

	for pair_idx, (l1, l2) in enumerate(pairs):
		v1, v2 = data_dict[l1], data_dict[l2]
		valid_pair = ~np.isnan(v1) & ~np.isnan(v2)
		if valid_pair.sum() < 5: continue
		try:
			_, p = wilcoxon(v1[valid_pair], v2[valid_pair])
			y = y_max + y_step * (1 + pair_idx * 2)
			_draw_sig_bracket(ax, positions[l1], positions[l2], y,
							  _significance_stars(p), y_step * 0.5)
		except Exception: pass

	ax.set_xticks(pos_list); ax.set_xticklabels(labels, fontsize=10)
	ax.set_ylabel(ylabel, fontsize=10); ax.set_title(title, fontsize=11)
	ax.grid(True, alpha=0.1, axis='y')


# =============================================================================
# COMPUTE PER-SESSION METRICS (actual simulated choices)
# =============================================================================

def compute_session_metrics(session_data, be_params, sc_params,
							 burn_in=1000, n_reps=20, seed=42):
	"""
	For each session, simulate BE and SC choices and compute:
	- choice_agreement: model choice == real choice (across reps)
	- task_accuracy: model choice == correct category (across reps)
	- um_mse: from session_data (already computed)
	"""
	from models.BE_core import BEParams, BEModel
	from models.SC_core import SCParams, SCModel

	be_p = BEParams(**be_params)
	sc_p = SCParams(**sc_params)
	metrics = []

	for d in session_data:
		stim, cat, real_ch = d['stimuli'], d['categories'], d['choices']
		valid_real = ~np.isnan(real_ch)

		m = {'session_idx': d['session_idx'], 'real_accuracy': d['accuracy'],
			 'be_um_mse': d['be_um_mse'], 'sc_um_mse': d['sc_um_mse'],
			 'n_trials': len(stim)}

		for model_label, model_params, model_cls, state_cls, seed_offset in [
			('be', be_p, BEModel, None, 0),
			('sc', sc_p, SCModel, None, 1000),
		]:
			agreements, task_accs = [], []
			for rep in range(n_reps):
				rng = np.random.default_rng(seed + rep + seed_offset)
				if model_label == 'be':
					state = BEModel.create_initial_state(params=be_p, burn_in=burn_in, seed=seed)
					ch, _, _, _ = BEModel.simulate_session(
						be_p, state, stim, cat, rng, return_history=False)
				else:
					state = SCModel.create_initial_state(params=sc_p, burn_in=burn_in, seed=seed)
					ch, _, _, _ = SCModel.simulate_session(
						sc_p, state, stim, cat, rng, return_history=False)

				both_valid = valid_real & ~np.isnan(ch)
				if both_valid.sum() > 0:
					agreements.append(np.mean(ch[both_valid] == real_ch[both_valid]))
					task_accs.append(np.mean(ch[both_valid] == cat[both_valid]))

			m[f'{model_label}_choice_agreement'] = np.mean(agreements) if agreements else np.nan
			m[f'{model_label}_task_accuracy'] = np.mean(task_accs) if task_accs else np.nan

		metrics.append(m)
	return metrics


# =============================================================================
# PER-ANIMAL BOXPLOTS (dots = sessions)
# =============================================================================

def plot_animal_summary_boxplots(animal_id, metrics,
								  be_colour='steelblue', sc_colour='darkorange'):
	"""3 panels: choice agreement, task accuracy, UM MSE. Dots = sessions."""
	if not metrics: print(f"No metrics for {animal_id}"); return

	real_acc = np.array([m['real_accuracy'] for m in metrics])
	be_agree = np.array([m['be_choice_agreement'] for m in metrics])
	sc_agree = np.array([m['sc_choice_agreement'] for m in metrics])
	be_task = np.array([m['be_task_accuracy'] for m in metrics])
	sc_task = np.array([m['sc_task_accuracy'] for m in metrics])
	be_mse = np.array([m['be_um_mse'] for m in metrics])
	sc_mse = np.array([m['sc_um_mse'] for m in metrics])

	fig, axes = plt.subplots(1, 3, figsize=(15, 5))

	_boxplot_with_dots_and_tests(
		axes[0], {'BE': be_agree, 'SC': sc_agree},
		{'BE': be_colour, 'SC': sc_colour}, {'BE': 0, 'SC': 1},
		['BE', 'SC'], [('BE', 'SC')],
		ylabel='Fraction', title='Choice Agreement\n(model == real)',
	)
	axes[0].axhline(0.5, color='grey', ls=':', alpha=0.4, lw=1)

	_boxplot_with_dots_and_tests(
		axes[1], {'Real': real_acc, 'BE': be_task, 'SC': sc_task},
		{'Real': 'black', 'BE': be_colour, 'SC': sc_colour},
		{'Real': 0, 'BE': 1, 'SC': 2}, ['Real', 'BE', 'SC'],
		[('Real', 'BE'), ('Real', 'SC'), ('BE', 'SC')],
		ylabel='Fraction correct', title='Task Accuracy\n(choice == correct)',
	)

	_boxplot_with_dots_and_tests(
		axes[2], {'BE': be_mse, 'SC': sc_mse},
		{'BE': be_colour, 'SC': sc_colour}, {'BE': 0, 'SC': 1},
		['BE', 'SC'], [('BE', 'SC')],
		ylabel='MSE (vs Real UM)', title='Update Matrix MSE',
	)

	um_winner = 'BE' if np.nanmedian(be_mse) < np.nanmedian(sc_mse) else 'SC'
	ag_winner = 'BE' if np.nanmedian(be_agree) > np.nanmedian(sc_agree) else 'SC'
	fig.suptitle(f'{animal_id} — {len(metrics)} sessions',
				 fontsize=13, fontweight='bold')
	plt.tight_layout(); plt.show()
	return fig


# =============================================================================
# GROUP BOXPLOTS (dots = animals)
# =============================================================================

def plot_group_summary_boxplots(all_animal_metrics, title='Group Summary',
								 be_colour='steelblue', sc_colour='darkorange',
								 summaries=None):
	"""
	Row 1: 3 boxplots (agreement, task accuracy, UM MSE). Dots = animals.
	Row 2: Mean update matrices (Real | BE | SC) averaged across animals.
	
	Args:
		all_animal_metrics: {animal_id: list of per-session metric dicts}
		summaries: from collect_animal_summaries (needed for UM row).
				   If None, only row 1 is shown.
	"""
	from behav_utils.plotting.update_matrix import plot_update_matrix as _plot_um

	aids = list(all_animal_metrics.keys())
	n = len(aids)
	has_um = summaries is not None and len(summaries) > 0
	n_rows = 2 if has_um else 1

	real_acc = np.array([np.nanmean([m['real_accuracy']
						 for m in all_animal_metrics[a]]) for a in aids])
	be_agree = np.array([np.nanmean([m['be_choice_agreement']
						 for m in all_animal_metrics[a]]) for a in aids])
	sc_agree = np.array([np.nanmean([m['sc_choice_agreement']
						 for m in all_animal_metrics[a]]) for a in aids])
	be_task = np.array([np.nanmean([m['be_task_accuracy']
						for m in all_animal_metrics[a]]) for a in aids])
	sc_task = np.array([np.nanmean([m['sc_task_accuracy']
						for m in all_animal_metrics[a]]) for a in aids])
	be_mse = np.array([np.nanmean([m['be_um_mse']
					   for m in all_animal_metrics[a]]) for a in aids])
	sc_mse = np.array([np.nanmean([m['sc_um_mse']
					   for m in all_animal_metrics[a]]) for a in aids])

	fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5.5 * n_rows))
	if n_rows == 1:
		axes = axes.reshape(1, -1)

	# ── Row 1: Boxplots ──────────────────────────────────────────────
	_boxplot_with_dots_and_tests(
		axes[0, 0], {'BE': be_agree, 'SC': sc_agree},
		{'BE': be_colour, 'SC': sc_colour}, {'BE': 0, 'SC': 1},
		['BE', 'SC'], [('BE', 'SC')],
		ylabel='Mean per animal', title='Choice Agreement\n(model == real)',
		dot_size=40,
	)
	axes[0, 0].axhline(0.5, color='grey', ls=':', alpha=0.4, lw=1)

	_boxplot_with_dots_and_tests(
		axes[0, 1], {'Real': real_acc, 'BE': be_task, 'SC': sc_task},
		{'Real': 'black', 'BE': be_colour, 'SC': sc_colour},
		{'Real': 0, 'BE': 1, 'SC': 2}, ['Real', 'BE', 'SC'],
		[('Real', 'BE'), ('Real', 'SC'), ('BE', 'SC')],
		ylabel='Mean per animal', title='Task Accuracy\n(choice == correct)',
		dot_size=40,
	)

	_boxplot_with_dots_and_tests(
		axes[0, 2], {'BE': be_mse, 'SC': sc_mse},
		{'BE': be_colour, 'SC': sc_colour}, {'BE': 0, 'SC': 1},
		['BE', 'SC'], [('BE', 'SC')],
		ylabel='Mean MSE per animal', title='Update Matrix MSE',
		dot_size=40,
	)

	# Label dots
	rng = np.random.default_rng(42)
	for ax_idx, arrays in [
		(0, [(0, be_agree), (1, sc_agree)]),
		(2, [(0, be_mse), (1, sc_mse)]),
	]:
		for pos, arr in arrays:
			valid = ~np.isnan(arr)
			jitter = rng.uniform(-0.1, 0.1, valid.sum())
			for j, idx in enumerate(np.where(valid)[0]):
				axes[0, ax_idx].annotate(
					aids[idx], (pos + jitter[j], arr[idx]),
					fontsize=5, alpha=0.5, ha='center', va='bottom')

	# ── Row 2: Mean update matrices ──────────────────────────────────
	if has_um:
		real_ums = [s['real_um'] for s in summaries
					if not np.all(np.isnan(s['real_um']))]
		be_ums = [s['be_um'] for s in summaries
				  if not np.all(np.isnan(s['be_um']))]
		sc_ums = [s['sc_um'] for s in summaries
				  if not np.all(np.isnan(s['sc_um']))]

		real_mean = np.nanmean(real_ums, axis=0) if real_ums else None
		be_mean = np.nanmean(be_ums, axis=0) if be_ums else None
		sc_mean = np.nanmean(sc_ums, axis=0) if sc_ums else None

		vlim = max(
			np.nanmax(np.abs(real_mean)) if real_mean is not None else 0,
			np.nanmax(np.abs(be_mean)) if be_mean is not None else 0,
			np.nanmax(np.abs(sc_mean)) if sc_mean is not None else 0,
		)

		be_um_mse = matrix_error(be_mean, real_mean) if (be_mean is not None and real_mean is not None) else np.nan
		sc_um_mse = matrix_error(sc_mean, real_mean) if (sc_mean is not None and real_mean is not None) else np.nan

		for col, (um, um_title) in enumerate([
			(real_mean, f'Real (n={len(real_ums)} animals)'),
			(be_mean, f'BE (Mean UM MSE={be_um_mse:.5f})'),
			(sc_mean, f'SC (Mean UM MSE={sc_um_mse:.5f})'),
		]):
			ax = axes[1, col]
			if um is not None:
				_plot_um(um, ax=ax, vmin=-vlim, vmax=vlim)
			else:
				ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center')
			ax.set_title(um_title, fontsize=10)

	be_um_wins = np.sum(be_mse < sc_mse)
	sc_um_wins = np.sum(sc_mse < be_mse)
	fig.suptitle(f'{title}  ({n} animals) ',
				 fontsize=13, fontweight='bold')
	plt.tight_layout()
	plt.show()
	return fig

# =============================================================================
# PSYCHOMETRIC PARAM COMPARISON (BOXPLOT VERSION)
# =============================================================================

def plot_param_comparison_box(summaries, title='Psychometric Parameters: Real vs BE vs SC',
							   be_colour='steelblue', sc_colour='darkorange',
							   params_to_show=None):
	"""
	Row 1: boxplots of psychometric params (μ, σ, λ_A, λ_B)
	Row 2: psychometric curves (All overlay | Real | BE | SC)
	Each curve panel: thick mean + SEM band + thin individual lines.
	"""
	if params_to_show is None:
		params_to_show = ['mu', 'sigma', 'lapse_low', 'lapse_high']

	param_labels = {'mu': '\u03bc (PSE)', 'sigma': '\u03c3 (slope)',
					'lapse_low': '\u03bb_A (lapse)', 'lapse_high': '\u03bb_B (lapse)'}
	sources = ['Real', 'BE', 'SC']
	psych_keys = {'Real': 'real_psych', 'BE': 'be_psych', 'SC': 'sc_psych'}
	colours = {'Real': 'black', 'BE': be_colour, 'SC': sc_colour}
	positions = {'Real': 0, 'BE': 1, 'SC': 2}

	# Collect param values
	values = {p: {src: [] for src in sources} for p in params_to_show}
	for s in summaries:
		for src in sources:
			psych = s[psych_keys[src]]
			for p in params_to_show:
				values[p][src].append(
					psych.get(p, np.nan) if psych.get('success', False) else np.nan)
	for p in params_to_show:
		for src in sources:
			values[p][src] = np.array(values[p][src])

	# Collect curves
	x_fine = np.linspace(-1.1, 1.1, 200)
	all_curves = {src: [] for src in sources}
	for s in summaries:
		for src in sources:
			psych = s[psych_keys[src]]
			if psych.get('success', False):
				y = cumulative_gaussian(x_fine, psych['mu'], psych['sigma'],
										psych['lapse_low'], psych['lapse_high'])
				all_curves[src].append(y)
			else:
				all_curves[src].append(None)

	n_params = len(params_to_show)
	fig, axes = plt.subplots(2, n_params, figsize=(3.5 * n_params, 9))

	# ── Row 1: Parameter boxplots ─────────────────────────────────────
	for i, param in enumerate(params_to_show):
		_boxplot_with_dots_and_tests(
			axes[0, i], {src: values[param][src] for src in sources},
			colours, positions, sources,
			[('Real', 'BE'), ('Real', 'SC'), ('BE', 'SC')],
			ylabel='Value' if i == 0 else '',
			title=param_labels.get(param, param),
		)

	# ── Row 2: Psychometric curves ────────────────────────────────────
	# Panel configs: (title, which sources to show with thin lines, which with thick)
	curve_panels = [
		('All overlay', sources, sources),
		('Real only', ['Real'], ['Real']),
		('BE only', ['BE'], ['BE']),
		('SC only', ['SC'], ['SC']),
	]

	for col, (panel_title, thin_sources, thick_sources) in enumerate(curve_panels):
		if col >= n_params:
			break
		ax = axes[1, col]

		# Thin individual lines
		for src in thin_sources:
			for curve in all_curves[src]:
				if curve is not None:
					ax.plot(x_fine, curve, '-', color=colours[src],
							alpha=0.15, lw=0.8)

		# Thick mean + SEM
		for src in thick_sources:
			valid_curves = [c for c in all_curves[src] if c is not None]
			if valid_curves:
				mean_c = np.nanmean(valid_curves, axis=0)
				sem_c = np.nanstd(valid_curves, axis=0) / np.sqrt(len(valid_curves))
				ax.fill_between(x_fine, mean_c - sem_c, mean_c + sem_c,
								color=colours[src], alpha=0.2)
				ax.plot(x_fine, mean_c, '-', color=colours[src], lw=2.5,
						label=f'{src} (n={len(valid_curves)})')

		ax.axhline(0.5, color='grey', ls='--', alpha=0.3, lw=0.5)
		ax.axvline(0, color='grey', ls='--', alpha=0.3, lw=0.5)
		ax.set_xlim(-1.15, 1.15)
		ax.set_ylim(-0.05, 1.05)
		ax.set_title(panel_title, fontsize=10)
		ax.legend(fontsize=7, loc='lower right')
		if col == 0:
			ax.set_ylabel('P(choose B)', fontsize=10)
		ax.set_xlabel('Stimulus', fontsize=9)

	fig.suptitle(title, fontsize=14, fontweight='bold')
	plt.tight_layout()
	plt.show()
	return fig


In [ ]:
results_to_use = p1

summaries = collect_animal_summaries_2(
	results_to_use, all_session_data,
	n_bins=8, n_bootstrap=200,
)
print(f"Collected summaries for {len(summaries)} animals")
path = '/Users/Serkan/Downloads/sbi_expert_heuristics'

In [ ]:
# ── Figure 1: Animal grid (Real UM | BE UM | SC UM | Psychometrics) ──
fig = plot_animal_grid(summaries, title='Expert Sessions — Model Comparison')
fig.savefig(fname=f'{path}/animal_update_matrix.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# ── Psychometric params: boxplots with paired dots (replaces bar version) ──
fig = plot_param_comparison_box(summaries, title='Psychometric Parameters')
fig.savefig(fname=f'{path}/summary_psychometry.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# ── Compute metrics (simulates actual choices — ~30 sec per animal) ───
all_animal_metrics = {}

for r in results_to_use:  # p1 or p2
	aid = r['animal_id']
	sdata = all_session_data[aid]
	print(f"Computing metrics for {aid}...")
	
	metrics = compute_session_metrics(
		sdata, r['be_params'], r['sc_params'],
		burn_in=EXPERT_BURN_IN, n_reps=20, seed=SEED,
	)
	all_animal_metrics[aid] = metrics

In [ ]:
# ── Per-animal boxplots (dots = sessions) ─────────────────────────────
for aid, metrics in all_animal_metrics.items():
	fig = plot_animal_summary_boxplots(aid, metrics)
	fig.savefig(fname=f'{path}/animals/{aid}_summary.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# ── Group boxplots (dots = animals) ───────────────────────────────────
fig = plot_group_summary_boxplots(all_animal_metrics, title='Expert Sessions',
								  summaries=summaries)
fig.savefig(fname=f'{path}/model_summary.pdf', dpi=300, bbox_inches='tight')

In [ ]:
results_to_use = p2

summaries = collect_animal_summaries_2(
	results_to_use, all_session_data,
	n_bins=8, n_bootstrap=200,
)
print(f"Collected summaries for {len(summaries)} animals")
path = '/Users/Serkan/Downloads/sbi_expert_updateMatrix'

In [ ]:
# ── Figure 1: Animal grid (Real UM | BE UM | SC UM | Psychometrics) ──
fig = plot_animal_grid(summaries, title='Expert Sessions — Model Comparison')
fig.savefig(fname=f'{path}/animal_update_matrix.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# ── Psychometric params: boxplots with paired dots (replaces bar version) ──
fig = plot_param_comparison_box(summaries, title='Psychometric Parameters')
fig.savefig(fname=f'{path}/summary_psychometry.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# ── Compute metrics (simulates actual choices — ~30 sec per animal) ───
all_animal_metrics = {}

for r in results_to_use:  # p1 or p2
	aid = r['animal_id']
	sdata = all_session_data[aid]
	print(f"Computing metrics for {aid}...")
	
	metrics = compute_session_metrics(
		sdata, r['be_params'], r['sc_params'],
		burn_in=EXPERT_BURN_IN, n_reps=20, seed=SEED,
	)
	all_animal_metrics[aid] = metrics


In [ ]:
# ── Per-animal boxplots (dots = sessions) ─────────────────────────────
for aid, metrics in all_animal_metrics.items():
	fig = plot_animal_summary_boxplots(aid, metrics)
	fig.savefig(fname=f'{path}/animals/{aid}_summary.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# ── Group boxplots (dots = animals) ───────────────────────────────────
fig = plot_group_summary_boxplots(all_animal_metrics, title='Expert Sessions',
								  summaries=summaries)
fig.savefig(fname=f'{path}/model_summary.pdf', dpi=300, bbox_inches='tight')


In [ ]:
results_to_use = p3

summaries = collect_animal_summaries_2(
	results_to_use, all_session_data,
	n_bins=8, n_bootstrap=200,
)
print(f"Collected summaries for {len(summaries)} animals")
path = '/Users/Serkan/Downloads/sbi_naive_to_expert_heuristics'

In [ ]:
# ── Figure 1: Animal grid (Real UM | BE UM | SC UM | Psychometrics) ──
fig = plot_animal_grid(summaries, title='Expert Sessions — Model Comparison')
fig.savefig(fname=f'{path}/animal_update_matrix.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# ── Psychometric params: boxplots with paired dots (replaces bar version) ──
fig = plot_param_comparison_box(summaries, title='Psychometric Parameters')
fig.savefig(fname=f'{path}/summary_psychometry.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# ── Compute metrics (simulates actual choices — ~30 sec per animal) ───
all_animal_metrics = {}

for r in results_to_use:  # p1 or p2
	aid = r['animal_id']
	sdata = all_session_data[aid]
	print(f"Computing metrics for {aid}...")
	
	metrics = compute_session_metrics(
		sdata, r['be_params'], r['sc_params'],
		burn_in=EXPERT_BURN_IN, n_reps=20, seed=SEED,
	)
	all_animal_metrics[aid] = metrics

In [ ]:
# ── Per-animal boxplots (dots = sessions) ─────────────────────────────
for aid, metrics in all_animal_metrics.items():
	fig = plot_animal_summary_boxplots(aid, metrics)
	fig.savefig(fname=f'{path}/animals/{aid}_summary.pdf', dpi=300, bbox_inches='tight')

In [ ]:
# ── Group boxplots (dots = animals) ───────────────────────────────────
fig = plot_group_summary_boxplots(all_animal_metrics, title='Expert Sessions',
								  summaries=summaries)
fig.savefig(fname=f'{path}/model_summary.pdf', dpi=300, bbox_inches='tight')


# Validation

In [ ]:
from models.BE_core import BEParams, BEModel
from models.SC_core import SCParams, SCModel
from behav_utils.data.synthetic import sample_stimuli
from behav_utils.data.structures import FittingData
from behav_utils.analysis.update_matrix import compute_update_matrix, matrix_error
from behav_utils.analysis.summary_stats import compute_summary_stats
from behav_utils.analysis.psychometry import fit_psychometric
from behav_utils.analysis.utils import cumulative_gaussian
from behav_utils.plotting.update_matrix import plot_update_matrix

from inference.comparison import simulate_all_sessions
from plotting.comparison import (
	plot_session_by_session_um,
	plot_session_by_session_psychometric,
	plot_pooled_um_comparison,
)

import torch

In [ ]:
# =============================================================================
# SYNTHETIC VALIDATION — supports both Part 1 and Part 2 networks
# =============================================================================

from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

VALIDATE_PART2 = True       # Set False to skip Part 2 validation
N_SBI_SIMS_VAL = 5000       # Sims for Part 2 per-animal training
N_TRIALS_SYN = 3000
N_REPS_UM = 20
TIMEOUT = 30                # Seconds before declaring 0% acceptance

BE_PARAM_SETS = [
	{'sigma_percep': 0.15, 'A_repulsion': 0.08, 'eta_learning': 0.35, 'eta_relax': 0.10},
	{'sigma_percep': 0.10, 'A_repulsion': 0.15, 'eta_learning': 0.50, 'eta_relax': 0.05},
	{'sigma_percep': 0.25, 'A_repulsion': 0.05, 'eta_learning': 0.20, 'eta_relax': 0.15},
]
SC_PARAM_SETS = [
	{'sigma_percep': 0.15, 'A_repulsion': 0.08, 'gamma': 0.92, 'sigma_update': 0.30},
	{'sigma_percep': 0.10, 'A_repulsion': 0.15, 'gamma': 0.80, 'sigma_update': 0.50},
	{'sigma_percep': 0.25, 'A_repulsion': 0.05, 'gamma': 0.95, 'sigma_update': 0.20},
]


def safe_condition_on_animal(snpe_result, fd, n_samples=200, timeout=TIMEOUT):
	"""Condition with timeout. Returns None if 0% acceptance."""
	import torch
	pooled = fd.pool()
	obs = compute_summary_stats(
		pooled['choices'], pooled['stimuli'], pooled['categories'],
		stat_names=snpe_result['stat_names'], return_dict=False,
	)
	obs = np.nan_to_num(obs, nan=0.0)
	x_obs = torch.tensor(obs, dtype=torch.float32)

	def _sample():
		return snpe_result['posterior'].sample((n_samples,), x=x_obs)

	with ThreadPoolExecutor(max_workers=1) as executor:
		future = executor.submit(_sample)
		try:
			samples = future.result(timeout=timeout).numpy()
		except (FuturesTimeout, Exception):
			return None

	return {
		'samples': samples,
		'median_params': {name: float(np.median(samples[:, i]))
						  for i, name in enumerate(snpe_result['param_names'])},
		'param_names': snpe_result['param_names'],
		'observed_stats': obs, 'accepted': True,
	}


def _make_fd(label, stim, cat, choices):
	valid = ~np.isnan(choices)
	return FittingData(
		animal_id=label, session_ids=['syn'], session_dates=[None],
		session_indices=np.array([0]),
		stimuli=[stim[valid]], categories=[cat[valid]],
		choices=[choices[valid]],
		no_response=[np.zeros(valid.sum(), dtype=bool)],
		not_blockstart=[np.concatenate([[False], np.ones(valid.sum()-1, dtype=bool)])],
		n_sessions=1, trials_per_session=np.array([valid.sum()]),
	)


def simulate_and_collect_um(model_type, params, stim, cat, n_reps=20, burn_in=1000, seed=42):
	"""Simulate model and return mean UM + all choices."""
	if model_type == 'be':
		mp = BEParams(**params)
		cls = BEModel
	else:
		mp = SCParams(**params)
		cls = SCModel

	ums, all_ch = [], []
	for rep in range(n_reps):
		rng = np.random.default_rng(seed + rep)
		state = cls.create_initial_state(params=mp, burn_in=burn_in, seed=seed)
		ch, _, _, _ = cls.simulate_session(mp, state, stim, cat, rng, return_history=False)
		all_ch.append(ch)
		v = ~np.isnan(ch)
		if v.sum() > 50:
			um, _, _ = compute_update_matrix(stim[v], ch[v], cat[v])
			ums.append(um)
	mean_um = np.nanmean(ums, axis=0) if ums else None
	return mean_um, all_ch


def run_validation(network_label, be_snpe, sc_snpe, syn_animals):
	"""Run full validation for one pair of networks."""

	print(f"\n{'#'*60}")
	print(f"# VALIDATION: {network_label}")
	print(f"{'#'*60}")

	# ── 1. Parameter recovery ─────────────────────────────────────────
	print(f"\n{'='*60}")
	print(f"1. PARAMETER RECOVERY ({network_label})")
	print(f"{'='*60}")

	recovery_data = []
	for sa in syn_animals:
		if sa['true_model'] == 'BE':
			cond = safe_condition_on_animal(be_snpe, sa['fd'])
		else:
			cond = safe_condition_on_animal(sc_snpe, sa['fd'])

		if cond is None:
			print(f"\n{sa['label']}: correct model REJECTED — this is a problem!")
			continue

		sa[f'cond_{network_label}'] = cond
		print(f"\n{sa['label']} (true: {sa['true_model']}):")
		for pname, true_val in sa['true_params'].items():
			rec_val = cond['median_params'].get(pname, np.nan)
			err = abs(true_val - rec_val)
			print(f"    {pname}: true={true_val:.3f} rec={rec_val:.3f} "
				  f"err={err:.3f} {'✓' if err < 0.1 else '✗'}")
			recovery_data.append({
				'animal': sa['label'], 'model': sa['true_model'],
				'param': pname, 'true': true_val, 'recovered': rec_val,
			})

	# Recovery scatter
	if recovery_data:
		rec_df = pd.DataFrame(recovery_data)
		unique_params = rec_df['param'].unique()
		n_p = len(unique_params)
		fig, axes = plt.subplots(1, n_p, figsize=(4 * n_p, 4))
		if n_p == 1: axes = [axes]
		for ax, pname in zip(axes, unique_params):
			sub = rec_df[rec_df['param'] == pname]
			for mt, col in [('BE', BE_COL), ('SC', SC_COL)]:
				ms = sub[sub['model'] == mt]
				if len(ms) > 0:
					ax.scatter(ms['true'], ms['recovered'], color=col, s=60,
							   label=mt, zorder=5, edgecolors='white')
			lims = [min(sub['true'].min(), sub['recovered'].min()) - 0.05,
					max(sub['true'].max(), sub['recovered'].max()) + 0.05]
			ax.plot(lims, lims, 'k--', alpha=0.3)
			ax.set_xlim(lims); ax.set_ylim(lims)
			ax.set_xlabel('True'); ax.set_ylabel('Recovered')
			ax.set_title(pname); ax.set_aspect('equal')
			ax.legend(fontsize=8); ax.grid(True, alpha=0.1)
		fig.suptitle(f'Parameter Recovery ({network_label})', fontsize=13, fontweight='bold')
		plt.tight_layout(); plt.show()

	print("\n" + "=" * 60)
	print("2. MODEL RECOVERY — CV")
	print("=" * 60)

	cv_results = []

	for sa in syn_animals:
		label = sa['label']
		expected = sa['true_model']
		print(f"\n{label} (true: {expected}):")

		model_cv = {}
		for model_name, snpe in [('BE', be_snpe_exp), ('SC', sc_snpe_exp)]:
			try:
				cv = cv_um_comparison(snpe, sa['fd'], n_repeats=32,
									n_posterior_samples=20, seed=42)
				model_cv[model_name] = cv
				print(f"  {model_name}: mean MSE = {cv['mean_error']:.5f} "
					f"({len(cv['test_errors'])} valid repeats)")
			except Exception as e:
				# If posterior rejection causes issues, assign worst possible score
				model_cv[model_name] = {
					'test_errors': np.array([np.inf] * 32),
					'mean_error': np.inf, 'std_error': 0.0,
				}
				print(f"  {model_name}: FAILED (posterior rejection) — assigned inf MSE")

		comp = compare_models(model_cv['BE'], model_cv['SC'])
		correct = comp['winner'] == expected

		print(f"  → winner: {comp['winner']} (p={comp['p_value']:.3g}) "
			f"{'✓' if correct else '✗'}")

		cv_results.append({
			'label': label, 'expected': expected, 'got': comp['winner'],
			'correct': correct, 'p': comp['p_value'],
			'be_mean': comp['be_mean'], 'sc_mean': comp['sc_mean'],
			'be_cv': model_cv['BE'], 'sc_cv': model_cv['SC'], 'comp': comp,
		})

		# Only plot if both models have valid results
		if model_cv['BE']['mean_error'] < np.inf and model_cv['SC']['mean_error'] < np.inf:
			plot_cv_comparison(model_cv['BE'], model_cv['SC'], comp, label,
							f'(true: {expected})')
		else:
			print(f"  Skipping CV plot — wrong model was rejected (this is correct behaviour)")

	# Summary table
	cv_df = pd.DataFrame([{
		'animal': r['label'], 'true': r['expected'], 'assigned': r['got'],
		'correct': '✓' if r['correct'] else '✗', 'p': r['p'],
		'BE_err': r['be_mean'], 'SC_err': r['sc_mean'],
	} for r in cv_results])

	print(f"\n{'='*60}")
	print("MODEL RECOVERY SUMMARY")
	print(f"{'='*60}")
	print(cv_df.to_string(index=False))
	n_correct = sum(r['correct'] for r in cv_results)
	print(f"\nRecovery rate: {n_correct}/{len(cv_results)} "
		f"({100*n_correct/len(cv_results):.0f}%)")

	# ── 3. UM + psychometric recovery ─────────────────────────────────
	print(f"\n{'='*60}")
	print(f"3. UM + PSYCHOMETRIC RECOVERY ({network_label})")
	print(f"{'='*60}")

	n_syn = len(syn_animals)
	fig, axes = plt.subplots(n_syn, 5, figsize=(20, 3.5 * n_syn), squeeze=False)
	x_fine = np.linspace(-1.1, 1.1, 200)

	for row, sa in enumerate(syn_animals):
		valid = ~np.isnan(sa['choices'])
		s, c, cat = sa['stim'][valid], sa['choices'][valid], sa['cat'][valid]

		real_um, _, _ = compute_update_matrix(s, c, cat)
		real_psych = fit_psychometric(s, c)

		# Condition BOTH posteriors
		be_cond = safe_condition_on_animal(be_snpe, sa['fd'], timeout=TIMEOUT)
		sc_cond = safe_condition_on_animal(sc_snpe, sa['fd'], timeout=TIMEOUT)

		for name, cond in [('BE', be_cond), ('SC', sc_cond)]:
			if cond is None:
				expected_reject = name != sa['true_model']
				print(f"  {sa['label']}: {name} rejected — "
					  f"{'expected' if expected_reject else 'UNEXPECTED!'}")

		# Simulate BE
		if be_cond is not None:
			be_mean_um, be_choices_all = simulate_and_collect_um(
				'be', be_cond['median_params'], s, cat, N_REPS_UM)
			be_mse = matrix_error(be_mean_um, real_um) if be_mean_um is not None else np.nan
		else:
			be_mean_um, be_choices_all, be_mse = None, [], np.nan

		# Simulate SC
		if sc_cond is not None:
			sc_mean_um, sc_choices_all = simulate_and_collect_um(
				'sc', sc_cond['median_params'], s, cat, N_REPS_UM)
			sc_mse = matrix_error(sc_mean_um, real_um) if sc_mean_um is not None else np.nan
		else:
			sc_mean_um, sc_choices_all, sc_mse = None, [], np.nan

		# Colour scale
		vlim = np.nanmax(np.abs(real_um))
		if be_mean_um is not None:
			vlim = max(vlim, np.nanmax(np.abs(be_mean_um)))
		if sc_mean_um is not None:
			vlim = max(vlim, np.nanmax(np.abs(sc_mean_um)))

		# Winner
		if np.isnan(be_mse) and np.isnan(sc_mse): winner = 'neither'
		elif np.isnan(be_mse): winner = 'SC'
		elif np.isnan(sc_mse): winner = 'BE'
		else: winner = 'BE' if be_mse < sc_mse else 'SC'
		correct = winner == sa['true_model']

		# ── Col 0: Ground truth ──
		plot_update_matrix(real_um, ax=axes[row, 0], vmin=-vlim, vmax=vlim, show_colorbar=False)
		axes[row, 0].set_ylabel(f"{sa['label']}\n(true: {sa['true_model']})",
								 fontsize=10, fontweight='bold')
		if row == 0:
			axes[row, 0].set_title('Ground Truth', fontsize=11, fontweight='bold')

		# ── Col 1: BE fit ──
		if be_mean_um is not None:
			plot_update_matrix(be_mean_um, ax=axes[row, 1], vmin=-vlim, vmax=vlim, show_colorbar=False)
			axes[row, 1].text(0.02, 0.98, f'MSE={be_mse:.5f}',
							   transform=axes[row, 1].transAxes, fontsize=8, va='top', color=BE_COL)
		else:
			axes[row, 1].text(0.5, 0.5, 'Rejected\n(0% acceptance)',
							   transform=axes[row, 1].transAxes, ha='center', va='center',
							   fontsize=10, color=BE_COL, fontweight='bold')
			axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])
		if row == 0:
			axes[row, 1].set_title('BE fit', fontsize=11, fontweight='bold', color=BE_COL)

		# ── Col 2: SC fit ──
		if sc_mean_um is not None:
			plot_update_matrix(sc_mean_um, ax=axes[row, 2], vmin=-vlim, vmax=vlim, show_colorbar=False)
			axes[row, 2].text(0.02, 0.98, f'MSE={sc_mse:.5f}',
							   transform=axes[row, 2].transAxes, fontsize=8, va='top', color=SC_COL)
		else:
			axes[row, 2].text(0.5, 0.5, 'Rejected\n(0% acceptance)',
							   transform=axes[row, 2].transAxes, ha='center', va='center',
							   fontsize=10, color=SC_COL, fontweight='bold')
			axes[row, 2].set_xticks([]); axes[row, 2].set_yticks([])
		if row == 0:
			axes[row, 2].set_title('SC fit', fontsize=11, fontweight='bold', color=SC_COL)

		# ── Col 3: Residual ──
		if winner in ('BE', 'SC'):
			winner_um = be_mean_um if winner == 'BE' else sc_mean_um
			if winner_um is not None:
				plot_update_matrix(real_um - winner_um, ax=axes[row, 3],
								   vmin=-vlim, vmax=vlim, show_colorbar=False)
			else:
				axes[row, 3].text(0.5, 0.5, 'N/A', transform=axes[row, 3].transAxes,
								   ha='center', va='center')
				axes[row, 3].set_xticks([]); axes[row, 3].set_yticks([])
		else:
			axes[row, 3].text(0.5, 0.5, 'Both rejected', transform=axes[row, 3].transAxes,
							   ha='center', va='center')
			axes[row, 3].set_xticks([]); axes[row, 3].set_yticks([])
		if row == 0:
			axes[row, 3].set_title('Residual\n(truth − winner)', fontsize=11, fontweight='bold')

		# ── Col 4: Psychometric ──
		ax = axes[row, 4]
		if real_psych['success']:
			y = cumulative_gaussian(x_fine, real_psych['mu'], real_psych['sigma'],
									real_psych['lapse_low'], real_psych['lapse_high'])
			ax.plot(x_fine, y, 'k-', lw=2.5, label='Truth')

		if be_choices_all:
			be_psych = fit_psychometric(s, np.nanmean(be_choices_all, axis=0))
			if be_psych['success']:
				y = cumulative_gaussian(x_fine, be_psych['mu'], be_psych['sigma'],
										be_psych['lapse_low'], be_psych['lapse_high'])
				ax.plot(x_fine, y, '-', color=BE_COL, lw=1.8, label='BE')
		else:
			ax.text(0.5, 0.7, 'BE rejected', transform=ax.transAxes,
					ha='center', fontsize=8, color=BE_COL, alpha=0.6)

		if sc_choices_all:
			sc_psych = fit_psychometric(s, np.nanmean(sc_choices_all, axis=0))
			if sc_psych['success']:
				y = cumulative_gaussian(x_fine, sc_psych['mu'], sc_psych['sigma'],
										sc_psych['lapse_low'], sc_psych['lapse_high'])
				ax.plot(x_fine, y, '-', color=SC_COL, lw=1.8, label='SC')
		else:
			ax.text(0.5, 0.3, 'SC rejected', transform=ax.transAxes,
					ha='center', fontsize=8, color=SC_COL, alpha=0.6)

		ax.axhline(0.5, color='grey', ls='--', alpha=0.3, lw=0.5)
		ax.axvline(0, color='grey', ls='--', alpha=0.3, lw=0.5)
		ax.set_xlim(-1.15, 1.15); ax.set_ylim(-0.05, 1.05)
		ax.tick_params(labelsize=7)
		if row == 0:
			ax.set_title('Psychometric', fontsize=11, fontweight='bold')
			ax.legend(fontsize=7, loc='lower right')

		mark = '✓' if correct else '✗'
		col = 'green' if correct else 'red'
		txt = f'winner: {winner} {mark}' if winner != 'neither' else 'both rejected'
		ax.text(0.98, 0.02, txt, transform=ax.transAxes, fontsize=9,
				ha='right', va='bottom', fontweight='bold', color=col)

	fig.suptitle(f'Synthetic Validation: {network_label}', fontsize=14, fontweight='bold')
	plt.tight_layout(); plt.show()

	# ── Summary ───────────────────────────────────────────────────────
	print(f"\n{'='*60}")
	print(f"SUMMARY ({network_label})")
	print(f"{'='*60}")
	print(cv_df.to_string(index=False))
	print(f"\nModel recovery: {n_correct}/{len(cv_results)} "
		  f"({100*n_correct/len(cv_results):.0f}%)")

	return cv_df, recovery_data



In [ ]:

# =============================================================================
# Generate synthetic animals
# =============================================================================
print("Generating synthetic data...")
syn_animals = []

for i, params in enumerate(BE_PARAM_SETS):
	stim, cat = sample_stimuli(N_TRIALS_SYN, 'uniform', np.random.default_rng(100 + i))
	be_p = BEParams(**params)
	state = BEModel.create_initial_state(params=be_p, burn_in=1000, seed=100 + i)
	choices, _, _, _ = BEModel.simulate_session(
		be_p, state, stim, cat, np.random.default_rng(100 + i))
	syn_animals.append({
		'label': f'BE_{i+1}', 'true_model': 'BE', 'true_params': params,
		'stim': stim, 'cat': cat, 'choices': choices,
		'fd': _make_fd(f'BE_{i+1}', stim, cat, choices),
	})

for i, params in enumerate(SC_PARAM_SETS):
	stim, cat = sample_stimuli(N_TRIALS_SYN, 'uniform', np.random.default_rng(200 + i))
	sc_p = SCParams(**params)
	state = SCModel.create_initial_state(params=sc_p, burn_in=1000, seed=200 + i)
	choices, _, _, _ = SCModel.simulate_session(
		sc_p, state, stim, cat, np.random.default_rng(200 + i))
	syn_animals.append({
		'label': f'SC_{i+1}', 'true_model': 'SC', 'true_params': params,
		'stim': stim, 'cat': cat, 'choices': choices,
		'fd': _make_fd(f'SC_{i+1}', stim, cat, choices),
	})

print(f"Generated {len(syn_animals)} synthetic animals")

In [ ]:
from joblib import Parallel, delayed
from inference.simulator import create_be_simulator, create_sc_simulator, get_sbi_prior

N_JOBS = 8

def _sim_batch(theta_batch, seeds, model_type, stimuli, categories, stat_names, burn_in):
	"""Run a batch of simulations in one worker."""
	creator = create_be_simulator if model_type == 'be' else create_sc_simulator
	sim = creator(stimuli, categories, stat_names=stat_names, burn_in=burn_in)
	results = []
	for theta_1d, seed in zip(theta_batch, seeds):
		results.append(sim(theta_1d, seed=seed))
	return results


def train_per_animal_snpe_parallel(
	model_type, fitting_data, stat_names,
	n_simulations=10_000, burn_in=1000, seed=42, n_jobs=4,
):
	"""Per-animal SNPE with batched parallel simulations."""
	import torch, time
	from sbi.inference import SNPE

	name = model_type.upper()
	aid = fitting_data.animal_id
	pooled = fitting_data.pool()
	stim, cat = pooled['stimuli'], pooled['categories']

	print(f"  Training [{name}] for {aid} "
		  f"({n_simulations:,} sims, {n_jobs} workers)...")

	creator = create_be_simulator if model_type == 'be' else create_sc_simulator
	sim = creator(stim, cat, stat_names=stat_names, burn_in=burn_in)
	prior = get_sbi_prior(sim)

	t0 = time.time()
	theta = prior.sample((n_simulations,))
	theta_np = theta.numpy()

	batch_size = n_simulations // n_jobs
	batches = []
	for j in range(n_jobs):
		start = j * batch_size
		end = start + batch_size if j < n_jobs - 1 else n_simulations
		batches.append((theta_np[start:end], list(range(start, end))))

	print(f"    Simulating ({n_jobs} batches of ~{batch_size})...")
	batch_results = Parallel(n_jobs=n_jobs, backend='loky')(
		delayed(_sim_batch)(
			bt, bs, model_type, stim, cat, stat_names, burn_in,
		)
		for bt, bs in batches
	)

	all_results = []
	for batch in batch_results:
		all_results.extend(batch)
	x = torch.tensor(np.stack(all_results), dtype=torch.float32)

	valid = ~torch.any(torch.isnan(x), dim=1)
	n_valid = valid.sum().item()
	print(f"    {n_valid}/{n_simulations} valid ({100*n_valid/n_simulations:.0f}%)")

	inference = SNPE(prior=prior)
	inference.append_simulations(theta[valid], x[valid])
	posterior = inference.build_posterior(inference.train())

	dt = time.time() - t0
	print(f"    Done in {dt/60:.1f} min")

	return {
		'posterior': posterior, 'prior': prior,
		'param_names': sim.get_param_names(),
		'model_type': model_type, 'stat_names': stat_names,
		'burn_in': burn_in, 'training_time': dt, 'n_valid': n_valid,
	}

In [ ]:
# =============================================================================
# Run validation with Part 1 networks
# =============================================================================
cv_df_p1, rec_p1 = run_validation(
	'Part 1 (heuristics)',
	be_snpe_exp, sc_snpe_exp, syn_animals,
)

In [ ]:
# =============================================================================
# Run validation with Part 2 networks (per-animal, trained on synthetic data)
# =============================================================================
if VALIDATE_PART2:
	print("\n\nTraining Part 2 networks on synthetic data...")
	print("(1 BE + 1 SC synthetic animal, ~10 min each)")

	# Just validate on 1 BE and 1 SC to save time
	syn_subset = [syn_animals[0], syn_animals[3]]  # BE_1 and SC_1

	for sa in syn_subset:
		print(f"\nTraining networks for {sa['label']}...")
		sa['be_snpe_p2'] = train_per_animal_snpe_parallel(
			'be', sa['fd'], SBI_STATS_WITH_UM,
			n_simulations=N_SBI_SIMS_VAL, burn_in=1000,
			seed=42, n_jobs=N_JOBS,
		)
		sa['sc_snpe_p2'] = train_per_animal_snpe_parallel(
			'sc', sa['fd'], SBI_STATS_WITH_UM,
			n_simulations=N_SBI_SIMS_VAL, burn_in=1000,
			seed=43, n_jobs=N_JOBS,
		)
	# Run validation per synthetic animal with its own networks
	print(f"\n{'#'*60}")
	print(f"# VALIDATION: Part 2 (updateMatrix)")
	print(f"{'#'*60}")

	cv_results_p2 = []
	for sa in syn_subset:
		label = sa['label']
		expected = sa['true_model']
		print(f"\n{label} (true: {expected}):")

		model_cv = {}
		for model_name, snpe in [('BE', sa['be_snpe_p2']), ('SC', sa['sc_snpe_p2'])]:
			print(f"  Running CV for {model_name}...", end=' ', flush=True)
			try:
				cv = cv_um_comparison(snpe, sa['fd'], n_repeats=32, seed=42)
				model_cv[model_name] = cv
				print(f"done (MSE={cv['mean_error']:.5f})")
			except Exception:
				model_cv[model_name] = {
					'test_errors': np.array([np.inf] * 32),
					'mean_error': np.inf, 'std_error': 0.0,
				}
				print(f"FAILED (posterior rejection)")

		comp = compare_models(model_cv['BE'], model_cv['SC'])
		correct = comp['winner'] == expected
		print(f"  → winner: {comp['winner']} (p={comp['p_value']:.3g}) "
			f"{'✓' if correct else '✗'}")

		cv_results_p2.append({
			'label': label, 'expected': expected,
			'got': comp['winner'], 'correct': correct,
		})

		if model_cv['BE']['mean_error'] < np.inf and model_cv['SC']['mean_error'] < np.inf:
			plot_cv_comparison(model_cv['BE'], model_cv['SC'], comp, label,
							f"(Part 2, true: {expected})")
		else:
			print(f"  Skipping plot — wrong model rejected (correct behaviour)")

	n_correct_p2 = sum(r['correct'] for r in cv_results_p2)
	print(f"\nPart 2 recovery: {n_correct_p2}/{len(cv_results_p2)}")

In [ ]:


# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("GRAND VALIDATION SUMMARY")
print("=" * 60)
# print(f"\nPart 1 (heuristics):")
# print(cv_df_p1.to_string(index=False))

if VALIDATE_PART2:
	print(f"\nPart 2 (updateMatrix) — subset:")
	print(pd.DataFrame(cv_results_p2).to_string(index=False))

print("\nIf all ✓ → pipeline validated. Real-data results are credible.")